# TopoGen Earth — Inference across step counts (Flow Matching)

Compare generation quality with different numbers of ODE solver steps
using a pretrained checkpoint on OpenEarthMap data.

Relies on code from `src/`. Clone the repo alongside this notebook:
```
!git clone https://github.com/mihalko711/topogen-earth.git
%cd topogen-earth
```

In [ ]:
import os
import sys
sys.path.insert(0, "..")  # if running from notebooks/ subdir

import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader

from src.data.dataset import OpenEarthMapDataset
from src.models.config import UNetConfig
from src.models.model import create_unet

In [ ]:
DATA_PATH = (
    "/kaggle/input/aletbm/global-land-cover-mapping-openearthmap"
    if os.path.exists("/kaggle/input")
    else ""
)
# Path to a pretrained checkpoint (.pth)
CKPT_PATH = ""  # <-- FILL IN

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
print(f"DATA_PATH: {DATA_PATH!r}")
print(f"CKPT_PATH: {CKPT_PATH!r}")

## 1. Dataset — small subset, just for masks

In [ ]:
dataset = OpenEarthMapDataset(
    root_dir=DATA_PATH,
    split="train",
    crop_size=256,
    subset_size=20,
)

print(f"Dataset size: {len(dataset)}")

## 2. Model + checkpoint

In [ ]:
model_cfg = UNetConfig(
    sample_size=256,
    in_channels=6,
    out_channels=3,
    layers_per_block=2,
    block_out_channels=(32, 64, 128),
)
model = create_unet(model_cfg).to(device)

if CKPT_PATH and os.path.exists(CKPT_PATH):
    ckpt = torch.load(CKPT_PATH, map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"])
    print(f"Loaded checkpoint from epoch {ckpt.get('epoch', '?')}")
else:
    print("WARNING: No checkpoint loaded — model weights are random!")

model.eval()
print(f"Model params: {sum(p.numel() for p in model.parameters()):,}")

## 3. Helper: generation with fixed noise

Same Euler solver as `generate_steps`, but accepts a pre-drawn noise tensor
so all step counts start from the same noise for a fair comparison.

In [ ]:
@torch.no_grad()
def generate_with_fixed_noise(
    model: torch.nn.Module,
    mask: torch.Tensor,
    noise: torch.Tensor,
    num_steps: int = 10,
    device: str = "cuda",
    crop_size: int = 256,
) -> torch.Tensor:
    """Run Euler ODE solver starting from `noise`, return the final image."""
    x = noise.clone()
    mask = mask.to(device)
    if mask.dim() == 3:
        mask = mask.unsqueeze(0)
    elif mask.dim() == 5:
        mask = mask.squeeze(0)

    dt = 1.0 / num_steps
    for i in range(num_steps):
        t = torch.tensor([i * dt], device=device)
        model_input = torch.cat([x, mask], dim=1)
        velocity = model(model_input, t).sample
        x = x + velocity * dt
    return x.cpu()


def denormalize(t):
    return (t.permute(1, 2, 0) * 0.5 + 0.5).clamp(0, 1)

## 4. Inference on several masks

For each mask we fix one noise and generate final images with
`num_steps ∈ [1, 2, 5, 10, 20, 50, 100]`.

In [ ]:
NUM_STEPS_LIST = [1, 2, 5, 10, 20, 50, 100]
NUM_EXAMPLES = 3

rng = np.random.default_rng(42)
indices = rng.choice(len(dataset), size=NUM_EXAMPLES, replace=False)

results = []

for idx in indices:
    sample = dataset[int(idx)]
    mask = sample["mask"]
    target = sample["image"]

    noise = torch.randn((1, 3, 256, 256), device=device)

    gens = {}
    for ns in NUM_STEPS_LIST:
        gen = generate_with_fixed_noise(
            model, mask, noise, num_steps=ns, device=device, crop_size=256
        )
        gens[ns] = gen

    results.append({"mask": mask, "target": target, "gens": gens})

print(f"Generated {NUM_EXAMPLES} examples × {len(NUM_STEPS_LIST)} step counts")

## 5. Visualisation

Each example gets its own figure: 3 × 3 grid — mask, target, and 7 step counts.

In [ ]:
for ex_idx, res in enumerate(results):
    fig, axes = plt.subplots(3, 3, figsize=(12, 12))
    axes = axes.flatten()

    axes[0].imshow(denormalize(res["mask"]))
    axes[0].set_title("Input mask")
    axes[0].axis("off")

    axes[1].imshow(denormalize(res["target"]))
    axes[1].set_title("Target")
    axes[1].axis("off")

    for i, ns in enumerate(NUM_STEPS_LIST):
        img = res["gens"][ns]
        axes[2 + i].imshow(denormalize(img.squeeze(0)))
        axes[2 + i].set_title(f"steps = {ns}")
        axes[2 + i].axis("off")

    plt.suptitle(f"Example {ex_idx + 1} — dataset index {indices[ex_idx]}", y=1.02)
    plt.tight_layout()
    plt.show()

### Summary comparison

Compact view: rows = examples, columns = step counts.

In [ ]:
fig, axes = plt.subplots(
    NUM_EXAMPLES, len(NUM_STEPS_LIST) + 2, figsize=(3 * (len(NUM_STEPS_LIST) + 2), 3 * NUM_EXAMPLES)
)

for row, res in enumerate(results):
    axes[row, 0].imshow(denormalize(res["mask"]))
    axes[row, 0].set_title("Mask")
    axes[row, 0].axis("off")

    axes[row, 1].imshow(denormalize(res["target"]))
    axes[row, 1].set_title("Target")
    axes[row, 1].axis("off")

    for col, ns in enumerate(NUM_STEPS_LIST):
        img = res["gens"][ns]
        axes[row, col + 2].imshow(denormalize(img.squeeze(0)))
        axes[row, col + 2].set_title(f"{ns} steps")
        axes[row, col + 2].axis("off")

plt.tight_layout()
plt.show()